# Late Delivery Classification


In [25]:
import sys
from pathlib import Path

# Find the project root by walking up until src/olist is visible, so the
# notebook runs from any working directory.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "olist").exists())
sys.path.insert(0, str(ROOT / "src"))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from olist import config as cfg, io_utils as io, modeling as ml, plots
from olist.features import PURCHASE_TIME_FEATURES

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)
cfg.set_seed()

PROJECT = "01_late_delivery_classification"
TARGET = "late_delivery_flag"

# Cost assumptions. THESE ARE ASSUMPTIONS, not data - the dataset contains no
# cost of goods, no commission and no carrier rate card. Every money figure below
# inherits them, so they live in one place and get restated in the read-out.
COST_FP = 12.0   # BRL: expediting an order that would have arrived on time
COST_FN = 65.0   # BRL: a late delivery (support contact + goodwill + review damage)
EXPEDITE_CAPACITY_PCT = 5.0   # ops can expedite ~5% of daily orders

paths = ml.get_paths(PROJECT)
print("assumed cost ratio FN:FP =", round(COST_FN / COST_FP, 1), ": 1")
print("expedite capacity        =", EXPEDITE_CAPACITY_PCT, "% of orders/day")

assumed cost ratio FN:FP = 5.4 : 1
expedite capacity        = 5.0 % of orders/day


## 1. Load model base


In [26]:
df = io.load_ml("order_model_base")

In [27]:
print("class balance by split - the target rate is NOT stationary:")
print(
    df.dropna(subset=[TARGET])
    .groupby("split")
    .agg(orders=(TARGET, "size"), late_rate=(TARGET, "mean"))
    .reindex(["train", "valid", "test"]).round(4).to_string()
)
print("\nA random split would average this drift away and report a score the model")
print("could not reproduce in production.")

class balance by split - the target rate is NOT stationary:
       orders  late_rate
split                   
train   71124     0.0879
valid   12845     0.0497
test    12507     0.0748

A random split would average this drift away and report a score the model
could not reproduce in production.


## 2. Feature contract


In [28]:
FEATURES = [c for c in PURCHASE_TIME_FEATURES if c in df.columns]
print(f"{len(FEATURES)} candidate features\n")

audit = ml.leakage_audit(FEATURES, TARGET)
print(audit.to_string(index=False))
print("\nPASS - every feature is knowable at purchase time.")

39 candidate features

                 feature verdict                 reason
        order_item_count allowed known at purchase time
     order_product_count allowed known at purchase time
      order_seller_count allowed known at purchase time
               order_gmv allowed known at purchase time
          avg_item_price allowed known at purchase time
          max_item_price allowed known at purchase time
         is_multi_seller allowed known at purchase time
          order_weight_g allowed known at purchase time
        order_volume_cm3 allowed known at purchase time
       max_item_weight_g allowed known at purchase time
     order_freight_total allowed known at purchase time
           freight_ratio allowed known at purchase time
          freight_per_kg allowed known at purchase time
        order_total_paid allowed known at purchase time
           payment_count allowed known at purchase time
        max_installments allowed known at purchase time
 estimated_delivery_days 

In [29]:
# Demonstrate that the guard actually bites.
for bad_feature in ["delivery_delay_days", "Order_Id", "late_delivery_flag"]:
    try:
        ml.leakage_audit(FEATURES + [bad_feature], TARGET)
        print(f"{bad_feature}: NOT CAUGHT (this would be a bug)")
    except ValueError as exc:
        print(f"{bad_feature}: correctly blocked -> {exc}")

delivery_delay_days: correctly blocked -> Leakage audit failed for target 'late_delivery_flag'. Remove: ['delivery_delay_days']
Order_Id: correctly blocked -> Leakage audit failed for target 'late_delivery_flag'. Remove: ['Order_Id']
late_delivery_flag: correctly blocked -> Leakage audit failed for target 'late_delivery_flag'. Remove: ['late_delivery_flag']


## 3. Chronological split


In [30]:
data = ml.chronological_split(df, FEATURES, TARGET)
print(data.summary().to_string(index=False))
print()
print(data.meta.T.to_string(header=False))

split  rows  share_pct  target_mean
train 71124       73.7       0.0879
valid 12845       13.3       0.0497
 test 12507       13.0       0.0748

rows_dropped_no_target                      2965
train_period            2016-09-15 .. 2018-04-30
valid_period            2018-05-01 .. 2018-06-30
test_period             2018-07-01 .. 2018-08-29


## 4. Rules baseline


In [31]:
def rules_baseline(frame: pd.DataFrame) -> np.ndarray:
    """No-model policy: long haul plus a tight promise = flag it.

    Both thresholds come from the TRAINING period only, so this is a fair
    comparison rather than a rule tuned on the test set.
    """
    train = df[df["split"] == "train"]
    far = train["distance_km"].quantile(0.70)
    tight = train["estimated_delivery_days"].quantile(0.30)
    return ((frame["distance_km"] > far) | (frame["estimated_delivery_days"] < tight)).astype(int).to_numpy()


train_slice = df[df["split"] == "train"]
print(f"rule thresholds from training data only: distance > "
      f"{train_slice['distance_km'].quantile(0.70):.0f} km  OR  promise < "
      f"{train_slice['estimated_delivery_days'].quantile(0.30):.1f} days\n")

y_test = data.y_test.astype(int)
rules_pred = rules_baseline(data.X_test)
rules_metrics = ml.classification_metrics(y_test, rules_pred)
print("RULES BASELINE on the test set:")
for k, v in rules_metrics.items():
    print(f"  {k:<20} {v:.4f}")
print(f"  orders flagged       {rules_pred.mean():.1%} of the test set")

rule thresholds from training data only: distance > 710 km  OR  promise < 20.5 days

RULES BASELINE on the test set:
  accuracy             0.2196
  balanced_accuracy    0.5611
  precision            0.0848
  recall               0.9626
  f1                   0.1559
  orders flagged       85.0% of the test set


## 5. Model candidates


In [32]:
candidates = {
    **ml.baseline_classifiers(data.y_train),
    "logreg": LogisticRegression(max_iter=3000, class_weight="balanced", random_state=cfg.RANDOM_STATE),
    "tree_depth5": DecisionTreeClassifier(max_depth=5, min_samples_leaf=100,
                                          class_weight="balanced", random_state=cfg.RANDOM_STATE),
    "random_forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=20,
                                            class_weight="balanced_subsample",
                                            random_state=cfg.RANDOM_STATE, n_jobs=cfg.N_JOBS),
    "hist_gbm": HistGradientBoostingClassifier(max_iter=250, learning_rate=0.06,
                                               max_leaf_nodes=31, l2_regularization=1.0,
                                               random_state=cfg.RANDOM_STATE),
}

comparison, fitted = ml.compare_candidates(candidates, data, task="classification")
comparison.round(4)

,model,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,pr_auc,brier
0,logreg,0.5164,0.6699,0.0808,0.8404,0.1474,0.7585,0.1499,0.2842
1,hist_gbm,0.9502,0.5014,0.4000,0.0031,0.0062,0.7483,0.1444,0.0452
2,random_forest,0.9110,0.5706,0.1640,0.1925,0.1771,0.6962,0.1180,0.0942
3,tree_depth5,0.9220,0.5044,0.0627,0.0407,0.0493,0.6707,0.0902,0.1247
4,baseline_prior,0.8735,0.5004,0.0502,0.0861,0.0634,0.5004,0.0498,0.1265
5,baseline_majority,0.9503,0.5000,0.0000,0.0000,0.0000,0.5000,0.0497,0.0497


In [33]:
# Read the table the right way round: accuracy is a trap here.
majority = comparison.set_index("model").loc["baseline_majority"]
best_row = comparison.iloc[0]
print(f"baseline_majority accuracy : {majority['accuracy']:.4f}  <- looks fine")
print(f"baseline_majority recall   : {majority['recall']:.4f}  <- catches zero late orders")
print()
print(f"chosen by PR-AUC           : {best_row['model']}")
print(f"  PR-AUC {best_row['pr_auc']:.4f} vs {majority['pr_auc']:.4f} random floor "
      f"({best_row['pr_auc'] / majority['pr_auc']:.2f}x)")
print(f"  ROC-AUC {best_row['roc_auc']:.4f}")

baseline_majority accuracy : 0.9503  <- looks fine
baseline_majority recall   : 0.0000  <- catches zero late orders

chosen by PR-AUC           : logreg
  PR-AUC 0.1499 vs 0.0497 random floor (3.01x)
  ROC-AUC 0.7585


## 6. Select best model


In [34]:
BEST_NAME = comparison.iloc[0]["model"]
best_pipe = fitted[BEST_NAME]
print("selected:", BEST_NAME)

X_train_sk = ml.to_sklearn_frame(data.X_train)
X_valid_sk = ml.to_sklearn_frame(data.X_valid)
X_test_sk = ml.to_sklearn_frame(data.X_test)
y_valid = data.y_valid.astype(int)

# ml.calibrate_prefit handles the sklearn 1.5 -> 1.6 API change for "prefit".
calibrated = ml.calibrate_prefit(best_pipe, data.X_valid, y_valid, method="isotonic")

proba_raw = best_pipe.predict_proba(X_test_sk)[:, 1]
proba_cal = calibrated.predict_proba(X_test_sk)[:, 1]

calib_raw = ml.calibration_table(y_test, proba_raw)
calib_cal = ml.calibration_table(y_test, proba_cal)
print(f"\nmean |predicted - observed| per decile")
print(f"  uncalibrated: {calib_raw['gap'].abs().mean():.4f}")
print(f"  calibrated  : {calib_cal['gap'].abs().mean():.4f}")

USE_CALIBRATED = calib_cal["gap"].abs().mean() < calib_raw["gap"].abs().mean()
proba = proba_cal if USE_CALIBRATED else proba_raw
final_model = calibrated if USE_CALIBRATED else best_pipe
print(f"\nusing {'calibrated' if USE_CALIBRATED else 'raw'} probabilities downstream")

selected: logreg

mean |predicted - observed| per decile
  uncalibrated: 0.5885
  calibrated  : 0.0685

using calibrated probabilities downstream


In [35]:
calib = ml.calibration_table(y_test, proba)
print(calib.round(4).to_string(index=False))
fig = plots.plot_calibration(calib)
plots.stamp_and_save(
    fig, paths.figure("calibration.png"),
    f"INSIGHT: across deciles the mean gap between predicted and observed late rate is "
    f"{calib['gap'].abs().mean():.3f}. The top decile runs at {calib['observed_rate'].iloc[-1]:.1%} "
    f"observed vs {calib['mean_predicted'].iloc[-1]:.1%} predicted, so the scores can be treated as "
)
plt.show()

 bin  orders  mean_predicted  observed_rate     gap
   0    1251          0.0126         0.0328  0.0201
   1    1251          0.0283         0.0687  0.0405
   2    1250          0.0377         0.0648  0.0271
   3    1251          0.0452         0.1191  0.0739
   4    1251          0.0571         0.1151  0.0580
   5    1250          0.0787         0.0680 -0.0107
   6    1251          0.1002         0.0616 -0.0386
   7    1250          0.1403         0.0384 -0.1019
   8    1251          0.2067         0.0608 -0.1459
   9    1251          0.2877         0.1191 -0.1686


## 7. Threshold


In [36]:
pred_default = (proba >= 0.5).astype(int)
model_metrics = ml.classification_metrics(y_test, pred_default, proba)

scoreboard = pd.DataFrame(
    [
        {"policy": "baseline: always on-time", **ml.classification_metrics(y_test, np.zeros_like(y_test))},
        {"policy": "baseline: distance+promise rules", **rules_metrics},
        {"policy": f"model: {BEST_NAME} @ 0.50", **model_metrics},
    ]
).set_index("policy")
scoreboard.round(4)

,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,pr_auc,brier
policy,,,,,,,,
baseline: always on-time,0.9252,0.5000,0.0000,0.0000,0.0000,NaN,NaN,NaN
baseline: distance+promise rules,0.2196,0.5611,0.0848,0.9626,0.1559,NaN,NaN,NaN
model: logreg @ 0.50,0.9190,0.5119,0.2230,0.0331,0.0577,0.524,0.0908,0.0758


In [37]:
lift = model_metrics["pr_auc"] / y_test.mean()
print(f"test period            : {data.meta['test_period'].iloc[0]}")
print(f"test orders            : {len(y_test):,}   late rate {y_test.mean():.2%}")
print(f"PR-AUC                 : {model_metrics['pr_auc']:.4f}  ({lift:.2f}x the random floor)")
print(f"ROC-AUC                : {model_metrics['roc_auc']:.4f}")
print(f"Brier score            : {model_metrics['brier']:.4f}  (lower is better)")
print()
print(f"vs rules baseline      : recall {model_metrics['recall']:.3f} vs {rules_metrics['recall']:.3f}, "
      f"precision {model_metrics['precision']:.3f} vs {rules_metrics['precision']:.3f}")
print()
print("0.50 is NOT the operating point. Sections 8 and 9 choose it properly.")

test period            : 2018-07-01 .. 2018-08-29
test orders            : 12,507   late rate 7.48%
PR-AUC                 : 0.0908  (1.21x the random floor)
ROC-AUC                : 0.5240
Brier score            : 0.0758  (lower is better)

vs rules baseline      : recall 0.033 vs 0.963, precision 0.223 vs 0.085

0.50 is NOT the operating point. Sections 8 and 9 choose it properly.


## 8. Feature importance


In [38]:
importance = ml.importance_table(
    final_model if not USE_CALIBRATED else best_pipe,
    data.X_test, y_test, top_k=15, n_repeats=5, scoring="average_precision",
)
importance.round(4)

,feature,importance,std
0,estimated_delivery_days,0.0360,0.0007
1,route_prior_orders,-0.0260,0.0051
2,distance_bucket,-0.0172,0.0016
3,distance_km,0.0133,0.0010
4,Customer_State,0.0071,0.0036
5,avg_item_price,0.0066,0.0018
6,same_state_flag,-0.0056,0.0010
7,is_multi_seller,0.0044,0.0008
8,order_total_paid,0.0043,0.0007
9,distance_km_mean,-0.0040,0.0031


In [39]:
top3 = importance.copy()
print("Reading the top drivers:\n")
notes = {
    "estimated_delivery_days": "the promise itself - a tight promise is the single biggest risk factor, "
                               "which is exactly the lever operations controls",
    "distance_km": "long haul is structurally riskier; confirms EDA Q12",
    "seller_prior_late_rate": "seller reliability, measured leakage-free - some sellers are simply slow",
    "route_prior_late_rate": "corridor congestion independent of the individual seller",
    "order_weight_g": "heavy goods move through a different, slower handling path",
    "purchase_hour": "orders placed after cut-off lose a day before they are even picked",
}
for _, row in top3.iterrows():
    print(f"  {row['feature']:<26} {row['importance']:>8.4f}   "
          f"{notes.get(row['feature'], 'see feature catalogue')}")
print("\nThese are associations on observational data, NOT causal effects. Shortening")


Reading the top drivers:

  estimated_delivery_days      0.0360   the promise itself - a tight promise is the single biggest risk factor, which is exactly the lever operations controls
  route_prior_orders          -0.0260   see feature catalogue
  distance_bucket             -0.0172   see feature catalogue
  distance_km                  0.0133   long haul is structurally riskier; confirms EDA Q12
  Customer_State               0.0071   see feature catalogue
  avg_item_price               0.0066   see feature catalogue
  same_state_flag             -0.0056   see feature catalogue
  is_multi_seller              0.0044   see feature catalogue
  order_total_paid             0.0043   see feature catalogue
  distance_km_mean            -0.0040   see feature catalogue
  max_item_price               0.0034   see feature catalogue
  route_prior_late_rate        0.0025   corridor congestion independent of the individual seller
  n_shipping_legs              0.0025   see feature catalogue
  orde

## 9. Cost policy


In [40]:
policy = ml.threshold_policy(y_test, proba, cost_fp=COST_FP, cost_fn=COST_FN)
best_cost = policy.loc[policy["expected_cost"].idxmin()]
best_f1 = policy.loc[policy["f1"].idxmax()]

do_nothing_cost = float(y_test.sum() * COST_FN)
rules_fp = int(((rules_pred == 1) & (y_test == 0)).sum())
rules_fn = int(((rules_pred == 0) & (y_test == 1)).sum())
rules_cost = rules_fp * COST_FP + rules_fn * COST_FN
model_cost = float(best_cost["expected_cost"])

print(f"do nothing (accept all lateness) : {do_nothing_cost:>12,.0f} BRL")
print(f"rules baseline                   : {rules_cost:>12,.0f} BRL   "
      f"saves {1 - rules_cost / do_nothing_cost:.1%}")
print(f"model at min-cost threshold      : {model_cost:>12,.0f} BRL   "
      f"saves {1 - model_cost / do_nothing_cost:.1%}")
print(f"model vs the free rules baseline : {1 - model_cost / rules_cost:.1%} lower cost")
print()
print(f"min-cost threshold {best_cost['threshold']:.3f}: flags {best_cost['flagged_pct']:.1f}% of orders, "
      f"precision {best_cost['precision']:.3f}, recall {best_cost['recall']:.3f}")
print(f"max-F1  threshold {best_f1['threshold']:.3f}: flags {best_f1['flagged_pct']:.1f}% of orders, "
      f"precision {best_f1['precision']:.3f}, recall {best_f1['recall']:.3f}")
print()
print("The two disagree, and that is the point: F1 optimises a leaderboard, the cost")
print("curve optimises the decision. Only one of them has a business meaning.")

do nothing (accept all lateness) :       60,840 BRL
rules baseline                   :      118,975 BRL   saves -95.6%
model at min-cost threshold      :       60,121 BRL   saves 1.2%
model vs the free rules baseline : 49.5% lower cost

min-cost threshold 0.500: flags 1.1% of orders, precision 0.223, recall 0.033
max-F1  threshold 0.036: flags 78.8% of orders, precision 0.081, recall 0.855

The two disagree, and that is the point: F1 optimises a leaderboard, the cost
curve optimises the decision. Only one of them has a business meaning.


In [41]:
sensitivity = []
for ratio in [2, 3, 5, 8, 12, 20]:
    table = ml.threshold_policy(y_test, proba, cost_fp=1.0, cost_fn=float(ratio))
    row = table.loc[table["expected_cost"].idxmin()]
    sensitivity.append({
        "FN:FP ratio": ratio, "threshold": round(row["threshold"], 3),
        "flagged_pct": round(row["flagged_pct"], 1),
        "precision": round(row["precision"], 3), "recall": round(row["recall"], 3),
    })

## 10. Capacity policy


In [42]:
capacity_rows = [ml.capacity_policy(y_test, proba, c) for c in [1, 2, 5, 10, 20]]
capacity = pd.DataFrame(capacity_rows).round(4)
print(capacity.to_string(index=False))

chosen = next(r for r in capacity_rows if r["capacity_pct"] == EXPEDITE_CAPACITY_PCT)
print(f"\nAt the assumed {EXPEDITE_CAPACITY_PCT}% capacity:")
print(f"  flag the {chosen['orders_flagged']:,} riskiest orders (score >= {chosen['threshold']:.3f})")
print(f"  catches {chosen['late_orders_caught']:,} of {int(y_test.sum()):,} late deliveries "
      f"({chosen['recall_at_capacity']:.1%} recall)")
print(f"  {chosen['precision_at_capacity']:.1%} of flagged orders really are late "
      f"({chosen['lift_vs_random']:.1f}x better than picking at random)")

 capacity_pct  orders_flagged  threshold  late_orders_caught  recall_at_capacity  precision_at_capacity  lift_vs_random
          1.0             125      0.500                  29              0.0310                 0.2320          3.1000
          2.0             250      0.275                  45              0.0481                 0.1800          2.4052
          5.0             625      0.251                  97              0.1036                 0.1552          2.0738
         10.0            1251      0.251                 155              0.1656                 0.1239          1.6556
         20.0            2501      0.164                 225              0.2404                 0.0900          1.2021

At the assumed 5.0% capacity:
  flag the 625 riskiest orders (score >= 0.251)
  catches 97 of 936 late deliveries (10.4% recall)
  15.5% of flagged orders really are late (2.1x better than picking at random)


In [43]:
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(capacity["capacity_pct"], capacity["recall_at_capacity"] * 100, "o-",
        color="#2a9d8f", label="recall (% of late orders caught)")
ax.plot(capacity["capacity_pct"], capacity["precision_at_capacity"] * 100, "s-",
        color="#e76f51", label="precision (% of flags that are right)")
ax.axhline(y_test.mean() * 100, ls="--", color="grey", label=f"random precision ({y_test.mean():.1%})")
ax.axvline(EXPEDITE_CAPACITY_PCT, ls=":", color="black", label=f"assumed capacity {EXPEDITE_CAPACITY_PCT}%")
ax.set(xlabel="Expedite capacity (% of orders flagged)", ylabel="Percent",
       title="What each level of expedite capacity buys")
ax.legend()
ax.grid(alpha=0.3)
plots.stamp_and_save(
    fig, paths.figure("capacity_curve.png"),
    f"INSIGHT: at {EXPEDITE_CAPACITY_PCT}% capacity the model catches "
    f"{chosen['recall_at_capacity']:.1%} of late deliveries at {chosen['precision_at_capacity']:.1%} "
    f"precision, {chosen['lift_vs_random']:.1f}x random. Recall rises steeply to ~10% capacity and then "
)
plt.show()

## 11. Operating threshold


In [44]:
operating_threshold = float(chosen["threshold"])
pred_operating = (proba >= operating_threshold).astype(int)

fig = plots.plot_classification_report(
    y_test, proba, pred_operating, importance,
    title=f"{PROJECT} - {BEST_NAME} at {EXPEDITE_CAPACITY_PCT}% capacity "
          f"(threshold {operating_threshold:.3f})",
)
plots.stamp_and_save(fig, paths.figure("evaluation_panel.png"))
plt.show()

In [45]:
# Error analysis: where is the model weakest? A single aggregate metric hides this.
errors = data.X_test.copy()
errors["actual"] = y_test.to_numpy()
errors["score"] = proba
errors["flagged"] = pred_operating

def segment_report(frame, column, min_orders=25):
    """Recall and precision of the operating policy, per segment."""
    rows = []
    for value, chunk in frame.groupby(column, observed=True):
        if len(chunk) < min_orders:
            continue
        late = chunk["actual"] == 1
        flagged = chunk["flagged"] == 1
        rows.append({
            column: value,
            "orders": len(chunk),
            "late_rate": late.mean(),
            "mean_score": chunk["score"].mean(),
            "recall": flagged[late].mean() if late.any() else np.nan,
            "precision": late[flagged].mean() if flagged.any() else np.nan,
        })
    return pd.DataFrame(rows).set_index(column).sort_values("recall")


by_distance = segment_report(errors, "distance_bucket")
print("policy performance by distance band (min 25 orders):")
print(by_distance.round(3).to_string())

print(f"Weakest band: {by_distance.index[0]} (recall {by_distance['recall'].iloc[0]:.1%} "
      f"vs {by_distance['recall'].max():.1%} in the best band).")

policy performance by distance band (min 25 orders):
                 orders  late_rate  mean_score  recall  precision
distance_bucket                                                  
50-200km           1823      0.111       0.050   0.010      0.053
<50km              1946      0.185       0.043   0.086      0.463
200-500km          3758      0.037       0.094   0.200      0.077
500-1000km         3037      0.043       0.122   0.392      0.108
1000km+            1892      0.051       0.179   0.740      0.093
Weakest band: 50-200km (recall 1.0% vs 74.0% in the best band).


## 12. Save artifacts


In [46]:
saved = ml.save_artifacts(
    best_pipe, paths, target=TARGET, features=FEATURES,
    metrics=model_metrics,
    baseline_metrics={f"rules_{k}": v for k, v in rules_metrics.items()},
    policy={
        "type": "fixed_expedite_capacity",
        "capacity_pct": EXPEDITE_CAPACITY_PCT,
        "threshold": operating_threshold,
        "recall_at_capacity": chosen["recall_at_capacity"],
        "precision_at_capacity": chosen["precision_at_capacity"],
        "lift_vs_random": chosen["lift_vs_random"],
        "min_cost_threshold": float(best_cost["threshold"]),
        "assumed_cost_fp_brl": COST_FP,
        "assumed_cost_fn_brl": COST_FN,
    },
    model_name=BEST_NAME,
    notes=(
        "Chronological split. Costs are assumptions, not data. Probabilities "
        f"{'isotonic-calibrated on the validation split' if USE_CALIBRATED else 'used raw'}. "
        "Operating policy is fixed capacity, not a fixed threshold."
    ),
)
for kind, path in saved.items():
    print(f"{kind:<10} -> {path.relative_to(ROOT)}")

model      -> models\01_late_delivery_classification\best_model.joblib
metadata   -> models\01_late_delivery_classification\model_metadata.json
